# Build BGE-M3 FAISS index on a free Colab GPU

The CPU-only project machine needs ~23 h to embed 51k chunks; a free Colab T4 does it in ~20 min.

**How to use** (Runtime → Change runtime type → **T4 GPU**):
1. Run the setup cell.
2. Upload `data/processed/en_chunks.jsonl` (or `mk_chunks.jsonl`) when prompted.
3. Run the remaining cells; the last one downloads the finished `.index` file.
4. Put it at `data/indices/faiss/en_bge_m3.index` in the repo. Done — it is byte-compatible
   with `src.retrieval.dense_retriever.DenseRetriever` (IndexFlatIP over L2-normalized fp32,
   chunk-file order).

In [ ]:
!pip install -q FlagEmbedding==1.2.11 faiss-cpu
import torch
assert torch.cuda.is_available(), 'Switch runtime to a GPU (Runtime -> Change runtime type)'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload en_chunks.jsonl here
CHUNKS_FILE = next(iter(uploaded))
print('Using', CHUNKS_FILE)

In [ ]:
import json
texts = [json.loads(l)['text'] for l in open(CHUNKS_FILE, encoding='utf-8') if l.strip()]
print(len(texts), 'chunks')

from FlagEmbedding import BGEM3FlagModel
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, device='cuda')
out = model.encode(texts, batch_size=256, max_length=512)
embeddings = out['dense_vecs']
print('embeddings:', embeddings.shape)

In [ ]:
import faiss, numpy as np
emb = np.asarray(embeddings, dtype=np.float32)
faiss.normalize_L2(emb)
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

OUT = CHUNKS_FILE.replace('_chunks.jsonl', '_bge_m3.index')
faiss.write_index(index, OUT)
print('index:', index.ntotal, 'vectors ->', OUT)

# quick cross-lingual sanity check with a Macedonian query
q = model.encode(['Кој е главниот град на Македонија?'], batch_size=1, max_length=512)['dense_vecs']
q = np.asarray(q, dtype=np.float32); faiss.normalize_L2(q)
scores, ids = index.search(q, 3)
for s, i in zip(scores[0], ids[0]):
    print(f'{s:.3f}', texts[i][:110].replace('\n', ' '))

In [ ]:
files.download(OUT)